# AXI test 24: Gemini 3.8 Flash on ARC-AGI-3 games, alone and with AXI, counted the way a company that runs the model counts
**What this is.** Four public ARC-AGI-3 games (small puzzle games whose rules are not told), 60 moves each, played by **Gemini 3.8 Flash** in two ways: **alone** (one chat per game: every screen and every reply kept and sent again at every move) and **with AXI** (nothing kept in the chat; each move the model gets the current screen and a **Record the framework keeps exactly from the game's own screens**, and a check stops a move the game does not offer). Measured for both: **tokens** (prompt, cached, reply, thinking), **latency** (seconds per call and to the first token, by prompt size), **speed** (effective moves an hour), **wasted moves** (moves that did nothing, moves repeated that already did nothing), **illegal moves**, **loops**, and **levels completed**. The energy stand-in is computed tokens and seconds (the API gives no joules).

The predictions and the reading rules were written before any call in `docs/GEMINI_ARC_TEST.md` (hash-locked). **This is not a measure of how well the model solves new games and not a leaderboard entry**; a result of zero levels in both arms is a likely outcome, and the page says so beforehand.

**This test costs real money, and this notebook never turns billing on.** Steps: (1) type the prices in Cell 5; (2) run the pilot in Cell 6, which plays 12 moves of one game in each arm and prints what the whole test would cost **at your prices**; (3) only then type a budget in dollars (`BUDGET_USD`) and run Cells 7 and 8. The notebook refuses the next call once the saved answers cost the budget, and a second guard stops each run at a cap on tokens. Everything finished is saved and the same cell carries on.

Run each cell **one at a time, top to bottom**. **Keep this tab open** while a run is going. If it stops for any reason, open the notebook again, run Cells 2 to 5, then run the cell you were on again.

### Cell 1: your key (once; no code to run)
The Colab Secret `GOOGLE_API_KEY` with Notebook access on (you did this for the earlier notebooks; it is the same key). The ARC toolkit needs no key of its own: it takes an anonymous one by itself.

### Cell 2: get the code from GitHub (finish the update 77 commands in Codespaces and `git push` first)

In [ ]:
!rm -rf /content/axi && git clone -q --depth 1 https://github.com/gaurshalendra2412/axi-runtime /content/axi
import os
assert os.path.exists("/content/axi/python/axi/experiments/gemini_arc.py"), "the code is not on GitHub yet: finish the update 77 commands (git push) first"
print("code ok")

### Cell 3: install the Gemini library and the ARC toolkit (about 1 minute)

In [ ]:
!pip -q install -U google-genai arc-agi arcengine 2>&1 | tail -2

### Cell 4: connect Google Drive (a pop-up asks permission: Connect / Allow). Your progress and every reply are kept there

In [ ]:
import sys
sys.path.insert(0, "/content/axi/python")
from colab.colab_persist import open_store
store = open_store("/content/out", "axi_gemini_arc")

### Cell 5: settings and the games. Fill the lines at the top. PAID is True only after you turned billing on. The budget stays empty until you have read the pilot

In [ ]:
import json, time, logging
from google.colab import userdata, files
from axi.experiments import gemini_arc as G
from axi.experiments.api_llm import QuotaStop, ServiceBusy, BudgetStop

PAID = None          # True ONLY after you turned billing on for this key in Google AI Studio
PRICE_IN = None      # dollars per 1,000,000 INPUT tokens of Gemini 3.8 Flash
PRICE_OUT = None     # dollars per 1,000,000 OUTPUT tokens (thinking tokens are billed as output)
PRICE_CACHED = None  # dollars per 1,000,000 CACHED input tokens (type it: without it cached tokens are counted at the full price)
BUDGET_USD = None    # the most you allow this test to cost. Leave None until you have read the pilot in Cell 6
THINKING = None      # None = the model's own default

assert PAID in (True, False), "Set PAID: True if you turned billing on, False for a free key."
try: KEY = userdata.get("GOOGLE_API_KEY")
except Exception: raise SystemExit("No key found. Do Cell 1: a Secret named GOOGLE_API_KEY with Notebook access on.")
if PAID: assert PRICE_IN and PRICE_OUT, "Type PRICE_IN and PRICE_OUT (dollars per million tokens). The notebook has no built-in price on purpose."
TAG = str(THINKING or "default"); CACHE = "axi_gemini_arc_cache_%s.jsonl" % TAG
print("reply file:", store.restore(CACHE))
guard = dict(budget_usd=BUDGET_USD, price_in=PRICE_IN, price_out=PRICE_OUT, price_cached=PRICE_CACHED) if (PAID and BUDGET_USD) else dict(price_in=PRICE_IN, price_out=PRICE_OUT, price_cached=PRICE_CACHED)
llm, MODEL = G.GL.build_chat_llm(KEY, store.path(CACHE), daily_limit=None if PAID else 20, min_interval=0.3 if PAID else 4.5, on_write=lambda: store.persist(CACHE), thinking=THINKING, **guard)
META = dict(model=MODEL, thinking=THINKING)
sp = llm.spent_usd()
print("paid:", PAID, "| model:", MODEL, "| thinking:", THINKING or "model default", "| budget:", ("$%.2f" % BUDGET_USD) if BUDGET_USD else "NOT SET (pilot only)", "| spent so far with these settings:", ("$%.2f" % sp) if sp is not None else "-")
import arc_agi
logging.disable(logging.INFO)
ARC = arc_agi.Arcade()
def make_env(game):
    env = ARC.make(game, seed=G.GAME_SEED)
    assert env is not None, "could not make " + game
    return env
for g in G.GAMES: assert any(g == str(getattr(e, "game_id", e)) for e in ARC.get_environments()), g + " is not in the list of games the toolkit offers today"
print("games ready:", ", ".join(G.GAMES))
RESET = "The free quota resets at midnight Pacific time. Then open this notebook again, run Cells 2 to 5, and run this cell again."
BUSY = "This is Google's server being busy. Wait at least 30 minutes, then run THIS SAME CELL once. Do not run it again and again."
def go(fn):
    try: return fn()
    except QuotaStop as e:
        print("\nSTOPPED FOR NOW: " + str(e))
        if isinstance(e, BudgetStop): pass
        elif isinstance(e, ServiceBusy): print(BUSY)
        else: print(RESET)
        return None
def need_budget(): assert PAID and BUDGET_USD, "Read the pilot (Cell 6), then type BUDGET_USD in Cell 5, run Cell 5 again, and run this cell again."
def save(res, name):
    p = store.path(name + ".json"); json.dump(res, open(p, "w")); store.persist(name + ".json"); return p
def persisting(name, inner):
    print("progress file:", store.restore(name + ".jsonl"))
    def cb(*a):
        inner(*a)
        if a[0] % 10 == 0 or a[0] == a[1]: store.persist(name + ".jsonl")
    return cb
def prog(t, n, r):
    if t % 5 == 0 or t == n:
        sp = llm.spent_usd(); print(f"move {t}/{n} (game {r['game']} move {r['m']}) | prompt {r['prompt_tokens']:,} (cached {r['cache_tokens']:,}) | {r['seconds']:.1f} s | {r['kind']} legal {r['reply_ok']} changed {r['changed']} | levels {r['levels']}/{r['win_levels']} | spent {('$%.2f' % sp) if sp is not None else '-'}")

### Cell 6: the PILOT. 12 real moves of each arm in the first game (cents). It prints the real size of one screen in tokens, the speed and latency, and the projected cost of the whole test at your prices

In [ ]:
assert PAID, "The pilot needs a paid key (about 24 calls)."
pk = go(lambda: G.run(llm, make_env, "K", store.path("arc_pilot_k.jsonl"), games=(G.PILOT_GAME,), moves=G.PILOT_MOVES, meta=META))
pa = go(lambda: G.run(llm, make_env, "A", store.path("arc_pilot_a.jsonl"), games=(G.PILOT_GAME,), moves=G.PILOT_MOVES, meta=META))
if pk and pa:
    store.persist("arc_pilot_k.jsonl"); store.persist("arc_pilot_a.jsonl")
    G.print_stats("pilot K (kept chat)", pk["rows"]); G.print_stats("pilot A (with AXI)", pa["rows"]); G.print_games(pk["rows"]); G.print_games(pa["rows"])
    rep = G.pilot_report(pk["rows"], pa["rows"], PRICE_IN, PRICE_OUT, PRICE_CACHED)
    print()
    for k, v in rep.items(): print(f"  {k}: {v if not isinstance(v, float) else round(v, 3)}")
    print("\nRead these before the budget: k_first_prompt is the system text plus ONE screen (the real size of a screen in tokens is about that minus 500); k_growth_per_move is what each move adds to the kept chat; k_last_prompt_projected must reach 100,000 for reading rule 3 (if it does not, say so in the chat); cost_k is arm K with the cache share measured now (cost_k_nocache: no cache discount); cost_a is arm A.")
    print("Then type BUDGET_USD in Cell 5 and send me this printout before you run Cells 7 and 8.")

### Cell 7: ARM A, with AXI. 4 games x 60 moves. A line every 5 moves

In [ ]:
need_budget()
nameA = "arc_A_%s" % TAG
resA = go(lambda: G.run(llm, make_env, "A", store.path(nameA + ".jsonl"), meta=META, max_tokens=G.A_TOKEN_CAP, progress=persisting(nameA, prog)))
if resA:
    G.print_stats("ARM A (with AXI)", resA["rows"]); G.print_games(resA["rows"]); store.persist(nameA + ".jsonl")
    print("spent so far on this model: $%.2f" % llm.spent_usd()); files.download(save(resA, nameA))

### Cell 8: ARM K, the kept chat, alone. 4 games x 60 moves. The long and costly cell; if it stops (budget, busy server, tab closed), run Cells 2 to 5 and this cell again

In [ ]:
need_budget()
nameK = "arc_K_%s" % TAG
resK = go(lambda: G.run(llm, make_env, "K", store.path(nameK + ".jsonl"), meta=META, max_tokens=G.K_TOKEN_CAP, progress=persisting(nameK, prog)))
if resK:
    G.print_stats("ARM K (kept chat)", resK["rows"]); G.print_bands(resK["rows"]); G.print_games(resK["rows"]); store.persist(nameK + ".jsonl")
    print("spent so far on this model: $%.2f" % llm.spent_usd()); files.download(save(resK, nameK))

### Cell 9: the READING. Replays every saved move into fresh copies of the games (a check that nothing was changed), prints the tables and the 19 lines with yes / NO, and saves the report. Send me the output and the downloaded file

In [ ]:
ok = {}
try: G.replay(resK["rows"], make_env, "K"); ok["K"] = True
except Exception as e: print("replay K:", e); ok["K"] = False
try: G.replay(resA["rows"], make_env, "A"); ok["A"] = True
except Exception as e: print("replay A:", e); ok["A"] = False
checks = G.verdict(resK, resA, ok)
print("ARM K, by the size of the prompt:"); G.print_bands(resK["rows"])
print("\nARM K, by game:"); G.print_games(resK["rows"]); print("\nARM A, by game:"); G.print_games(resA["rows"])
G.print_verdict(checks)
try: sc = ARC.get_scorecard(); extra = str(sc)[:600] if sc else None
except Exception as e: extra = "scorecard unavailable: " + type(e).__name__
print("toolkit scorecard (shown, not used as a result):", extra)
print("usage:", llm.usage(), "| spent: $%.2f" % llm.spent_usd())
files.download(save(dict(K=resK, A=resA, checks=checks, replays=ok), "gemini_arc_all_%s" % TAG))

### Done
Files downloaded: the arm A, arm K and `gemini_arc_all_*.json` files and the `.jsonl` reply file. Upload `gemini_arc_all_default.json` to the chat. Nothing here says anything about any company's own service: it measures this model on these games.